In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, binom, t
from scipy.stats import ttest_1samp, ttest_ind, ttest_rel, levene, shapiro, probplot
import statsmodels.formula.api as smf

In [3]:
df = pd.read_csv("data/davis.csv")
s = df["height"]
q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
flag = df["height"].lt(low) | df["height"].gt(high)
display(df.loc[flag])
without_outliers = df.loc[~flag].copy()
print("Fences:", low, high)


,id,sex,weight,height,repwt,repht
11,12,F,166,57,56.0,163.0


Fences: 144.125 197.125


In [ ]:
count = 0
count_data = df.copy()
# print(count_data)
# wanted_column_names = ["repwt", "repht"]
# complete = count_data.dropna(subset=wanted_column_names)
# print(complete)
count_data[["repwt", "repht"]].corr()



,repwt,repht
repwt,1.00000,0.76186
repht,0.76186,1.00000


In [50]:
df_clean = pd.read_csv("data/davis_clean.csv")
renamed_by_position = df_clean.rename(
    columns={
        df_clean.columns[6]: "Error",
    }
)
df_clean["Report"] = 0
# for row in df_clean.itertuples():
#     if row[6] < 0:
#         row[7] = "Under"  
#     elif row[6] == 0:
#         row[7] = "Exact"
#     else:
#         row[7] = "Over"

df_clean["Report"] = pd.cut(df_clean["Error"], bins=[-10000, -0.5, 0, float("inf")], labels=["Under", "Exact", "Over"], include_lowest=True)

print(df_clean)

equal_count = 0
for row in df_clean.itertuples():
    if row[8] == "Equal":
        equal_count += 1

f_count = 0
s_count = 0
for row in df_clean.itertuples():
    if row[2] == "F":
        s_count += 1
        if row[8] == "Under":
            f_count += 1

print("equal: ", equal_count)
print("f: ", f_count)
print("s: ", s_count)

      id sex  weight  height  repwt  repht  Error Report
0      1   M      77     182   77.0  180.0    0.0  Exact
1      2   F      58     161   51.0  159.0   -7.0  Under
2      3   F      53     161   54.0  158.0    1.0   Over
3      4   M      68     177   70.0  175.0    2.0   Over
4      5   F      59     157   59.0  155.0    0.0  Exact
..   ...  ..     ...     ...    ...    ...    ...    ...
176  195   F      62     164   61.0  161.0   -1.0  Under
177  196   M      74     175   71.0  175.0   -3.0  Under
178  197   M      83     180   80.0  180.0   -3.0  Under
179  199   M      90     181   91.0  178.0    1.0   Over
180  200   M      79     177   81.0  178.0    2.0   Over

[181 rows x 8 columns]
equal:  0
f:  41
s:  99


In [ ]:
df_f = df_clean[(df_clean["sex"] == "F")]
df_m = df_clean[(df_clean["sex"] == "M")]

df_f.describe(include=None)





,id,weight,height,repwt,repht,Error
count,82.000000,82.000000,82.000000,82.000000,82.000000,82.000000
mean,95.231707,75.975610,178.012195,76.560976,176.256098,0.585366
std,61.397169,12.156364,6.649039,12.286797,6.977585,2.489242
min,1.000000,54.000000,163.000000,56.000000,161.000000,-5.000000
25%,42.500000,68.000000,173.000000,68.000000,171.250000,-1.000000
50%,89.000000,75.000000,178.000000,75.000000,175.000000,0.500000
75%,147.750000,83.000000,183.000000,83.000000,180.000000,2.000000
max,200.000000,119.000000,197.000000,124.000000,200.000000,9.000000


In [71]:
df = pd.read_csv("data/davis.csv")
mdl1 = smf.ols("height ~ weight", data=df).fit()
print("Coefficients:\n", mdl1.params.round(4))
print(
    f"\nR^2 = {mdl1.rsquared:.4f} (sqrt(R^2) = {np.sqrt(mdl1.rsquared):.4f}), Adj R^2 = {mdl1.rsquared_adj:.4f}"
)
print(
    f"Residual SE (sigma-hat) = {np.sqrt(mdl1.mse_resid):.3f} on df = {mdl1.df_resid:.0f}"
)
# print(
#     f"F-stat = {mdl1.fvalue:.2f} (p = {mdl1.f_pvalue:.3e}) == t_slope^2 = {mdl1.tvalues['Present_Price']**2:.2f}"
# )

Coefficients:
 Intercept    136.8305
weight         0.5170
dtype: float64

R^2 = 0.5946 (sqrt(R^2) = 0.7711), Adj R^2 = 0.5925
Residual SE (sigma-hat) = 5.702 on df = 198
